In [ ]:
import locale
def getpreferredencoding(do_setlocale = True):
    return "UTF-8"
locale.getpreferredencoding = getpreferredencoding

In [ ]:
!pip -q install configparser langchain openai
!pip -q install transformers huggingface_hub 
!pip -q install chromadb

In [ ]:
from langchain.llms.base import LLM
from langchain.llms.utils import enforce_stop_tokens
from langchain.llms import OpenAI
from langchain.tools import Tool
from langchain.agents import initialize_agent

In [ ]:
#Then convert the transformers pipeline into fact_llm 
import os
import dotenv
dotenv.load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
fact_llm = OpenAI(temperature=0)

In [ ]:
from langchain import PromptTemplate, HuggingFaceHub, LLMChain

template = """Question: {question}

Answer: Let's think step by step."""

prompt_open = PromptTemplate(template=template, input_variables=["question"])

open_chain = LLMChain(prompt=prompt_open,llm = fact_llm)

In [ ]:
question = "Given a source ontology and a target ontology. What is the matching class for source ontology class sensor?"

print(open_chain.run(question))

In [ ]:
from langchain.chains import RetrievalQA
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
recurSplitter = RecursiveCharacterTextSplitter(chunk_size=100,
                                               chunk_overlap=20,
                                               length_function=len)

In [ ]:
with open('initial.txt') as init:
  txt_init = init.read()

initial_docs = recurSplitter.create_documents([txt_init])

In [ ]:
with open('lexical.txt') as lin:
  txt_lin = lin.read()
lexical_docs = recurSplitter.create_documents([txt_lin])

In [ ]:
with open('graphical.txt') as spc:
  txt_spc = spc.read()
graphical_docs = recurSplitter.create_documents([txt_spc])

In [ ]:
from langchain.embeddings import OpenAIEmbeddings
embeddings = OpenAIEmbeddings()

In [ ]:
from langchain.vectorstores import FAISS
initial_chroma = FAISS.from_documents(initial_docs, embeddings)
lexical_chroma = FAISS.from_documents(lexical_docs, embeddings)
graphical_chroma = FAISS.from_documents(graphical_docs,embeddings)

In [ ]:
# Starting the custom tool route

In [ ]:
initial_retriever =RetrievalQA.from_chain_type(llm=fact_llm, 
                                           chain_type="stuff", 
                                           retriever=initial_chroma.as_retriever())

In [ ]:
lexical_retriever =RetrievalQA.from_chain_type(llm=fact_llm, 
                                           chain_type="stuff", 
                                           retriever=lexical_chroma.as_retriever())

In [ ]:
graphical_retriever =RetrievalQA.from_chain_type(llm=fact_llm, 
                                           chain_type="stuff", 
                                           retriever=graphical_chroma.as_retriever())

In [ ]:
tools = [
    Tool(
        name = "Initial",
        func=initial_retriever.run,
        description="useful for when you need to answer initial information."
    ),
    Tool(
        name = "Lexical",
        func=lexical_retriever.run,
        description="useful for when you need to answer lexical information."
    ),
    Tool(
        name = "Graphical",
        func=graphical_retriever.run,
        description="useful for when you need to answer graphical information."
    ),
]

In [ ]:
example_agent = initialize_agent(tools, 
                         fact_llm, 
                         agent="zero-shot-react-description", 
                         verbose=True)

In [ ]:
example_agent.run("What is the equivalent class for chairman in target ontology? Answer: First check the initial information. If it does not exist, retrive all the lexical and graphical information of chairman in the source ontology and retrieve all classes and their lexical and graphical information in the target ontology. Finally, find the most similar class to chairman in target ontology .")

In [ ]:
example_agent.run("What is the equivalent class for subject area in target ontology? Answer: First check the initial information. If it does not exist, retrive the information of subject area in the source ontology and retrieve all classes and their information in the target ontology. Finally, find the most similar class to subject area in target ontology .")

In [ ]:
example_agent.run("Please compare the similarity of the subject area in the source ontology with the paper in the target ontology. Think step by step. ")

In [ ]:
example_agent.run("You are acting as an Ontology Matcher. Please list all the entities that are similiar or equivalent in the source ontology and the target ontology. Think step by step. ")